# AstroCLIMB 03 — frozen representation cache

This Kaggle GPU notebook performs the expensive shared extraction step for experiments E03–E29. It encodes every Hugging Face row referenced by the canonical notebook 02 manifests and writes reusable float16 arrays indexed by `meta_row`.

Attach the private `astroclimb_01` and `astroclimb_02` Kaggle Datasets. Enable a GPU and Internet. The notebook streams the Hugging Face image split once and is resumable when the output directory from an earlier partial run is attached or restored.

In [1]:
%pip install -q "transformers>=4.51" "datasets>=3.0" scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import csv, gc, hashlib, io, json, os, pickle, shutil, time, traceback
import numpy as np
import torch
from PIL import Image, ImageOps
from scipy.fft import dctn
from datasets import load_dataset, Image as HFImage
from transformers import AutoImageProcessor, AutoModel, AutoProcessor, AutoTokenizer

HF_DATASET = 'adsabs/AstroCLIMB'
HF_SPLIT = 'train'
SPECTER_MODEL = 'allenai/specter2_base'
SIGLIP_MODEL = 'google/siglip2-base-patch16-naflex'
DINO_MODEL = 'facebook/dinov2-small'
TEXT_BATCH = 64
IMAGE_BATCH = 8
CHECKPOINT_EVERY = 512
RESUME_CACHE_DIR = None  # Optional attached directory from a failed/partial notebook 03 run
OUTPUT_DIR = Path('/kaggle/working/astroclimb_03') if Path('/kaggle/working').exists() else Path('results/astroclimb_03')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
HF_TOKEN = os.getenv('HF_TOKEN')
print({'device': DEVICE, 'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None, 'output': str(OUTPUT_DIR)})
assert DEVICE == 'cuda', 'Use a Kaggle GPU accelerator for this notebook.'

{'device': 'cuda', 'gpu': 'Tesla T4', 'output': '/kaggle/working/astroclimb_03'}


## Locate inputs and determine referenced rows

Only dictionary-format `metadata_index.pkl` files from notebook 01 are accepted. The two notebook 02 manifests determine the target rows.

In [3]:
def find_all(filename):
    roots = [Path('/kaggle/input'), Path('.'), Path('notebooks')]
    return sorted({p for root in roots if root.exists() for p in root.rglob(filename)}, key=lambda p: len(str(p)))

def load_index():
    failures = []
    candidates = sorted(find_all('metadata_index.pkl'), key=lambda p: ('astroclimb_01' not in str(p).lower(), len(str(p))))
    for path in candidates:
        try:
            with path.open('rb') as handle:
                value = pickle.load(handle)
            if isinstance(value, dict) and value.get('records') and 'caption' in value['records'][0]:
                return path, value
            failures.append(f'{path}: incompatible format')
        except Exception as exc:
            failures.append(f'{path}: {type(exc).__name__}')
    raise FileNotFoundError('Attach astroclimb_01. Candidates:\n' + '\n'.join(failures))

def choose_manifest(filename):
    candidates = sorted(find_all(filename), key=lambda p: ('astroclimb_02' not in str(p).lower(), len(str(p))))
    if not candidates:
        raise FileNotFoundError(f'Attach astroclimb_02; missing {filename}')
    return candidates[0]

INDEX_PATH, metadata_index = load_index()
TRAIN_MANIFEST = choose_manifest('hf_train_multimodal_pairs.csv')
VALIDATION_MANIFEST = choose_manifest('hf_validation_multimodal_pairs.csv')
records = metadata_index['records']
N_ROWS = max(int(r['meta_row']) for r in records) + 1
record_by_row = {int(r['meta_row']): r for r in records}
uuid_to_row = {str(r.get('uuid') or ''): int(r['meta_row']) for r in records if r.get('uuid')}

def manifest_rows(path):
    found = set()
    with path.open(newline='', encoding='utf-8') as handle:
        for row in csv.DictReader(handle):
            found.add(int(row['obj_1_row']))
            found.add(int(row['obj_2_row']))
    return found

train_rows = manifest_rows(TRAIN_MANIFEST)
validation_rows = manifest_rows(VALIDATION_MANIFEST)
target_rows = train_rows | validation_rows
assert train_rows.isdisjoint(validation_rows), 'Notebook 02 row leakage detected.'
assert target_rows.issubset(record_by_row)
print('Index:', INDEX_PATH)
print('Train manifest:', TRAIN_MANIFEST, 'unique rows:', len(train_rows))
print('Validation manifest:', VALIDATION_MANIFEST, 'unique rows:', len(validation_rows))
print('Total target rows:', len(target_rows), '/', N_ROWS, f'({len(target_rows)/N_ROWS:.1%})')

Index: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-01/astroclimb_01/metadata_index.pkl
Train manifest: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-02/astroclimb_02/hf_train_multimodal_pairs.csv unique rows: 72196
Validation manifest: /kaggle/input/datasets/syedmohaiminulhoque/astroclimb-02/astroclimb_02/hf_validation_multimodal_pairs.csv unique rows: 18254
Total target rows: 90450 / 94233 (96.0%)


## Resumable array helpers

A separate mask is the source of truth for completion; zero vectors are never interpreted as completed embeddings. Existing arrays are shape-checked before reuse.

In [4]:
def open_npy(name, shape, dtype, fill=0):
    path = OUTPUT_DIR / name
    if path.exists():
        array = np.load(path, mmap_mode='r+')
        if array.shape != tuple(shape) or array.dtype != np.dtype(dtype):
            raise ValueError(f'{path} has {array.shape}/{array.dtype}; expected {shape}/{np.dtype(dtype)}')
        return array
    array = np.lib.format.open_memmap(path, mode='w+', dtype=dtype, shape=shape)
    array[...] = fill
    array.flush()
    return array

def restore_partial_cache(source):
    if source is None:
        return
    source = Path(source)
    if not source.exists():
        raise FileNotFoundError(f'RESUME_CACHE_DIR does not exist: {source}')
    restored = []
    for item in source.iterdir():
        if item.is_file() and (item.suffix == '.npy' or item.name == 'extraction_errors.jsonl'):
            destination = OUTPUT_DIR / item.name
            if not destination.exists():
                shutil.copy2(item, destination)
                restored.append(item.name)
    print('Restored partial cache files:', restored)

restore_partial_cache(RESUME_CACHE_DIR)

target_mask = np.zeros(N_ROWS, dtype=bool)
target_mask[list(target_rows)] = True
target_mask_path = OUTPUT_DIR / 'target_mask.npy'
np.save(target_mask_path, target_mask)
errors_path = OUTPUT_DIR / 'extraction_errors.jsonl'

def log_error(stage, identifier, exc):
    with errors_path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps({'stage': stage, 'identifier': str(identifier), 'type': type(exc).__name__, 'message': str(exc)[:500]}) + '\n')

def normalized(array):
    if not torch.is_tensor(array):
        if hasattr(array, 'pooler_output') and array.pooler_output is not None:
            array = array.pooler_output
        elif hasattr(array, 'last_hidden_state'):
            array = array.last_hidden_state[:, 0]
        elif isinstance(array, (tuple, list)):
            array = array[0]
        else:
            raise TypeError(f'Cannot extract a tensor from {type(array)}')
    array = array.float()
    return array / array.norm(dim=-1, keepdim=True).clamp_min(1e-8)

def cleanup(*models):
    for model in models:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## SPECTER2 caption embeddings

The current project baseline uses the frozen base model's CLS representation. This preserves comparability with the previous hybrid notebook.

In [5]:
specter_tokenizer = AutoTokenizer.from_pretrained(SPECTER_MODEL, token=HF_TOKEN)
specter = AutoModel.from_pretrained(SPECTER_MODEL, token=HF_TOKEN, dtype=DTYPE).eval().to(DEVICE)
SPECTER_DIM = int(specter.config.hidden_size)
specter_text = open_npy('specter_text_f16.npy', (N_ROWS, SPECTER_DIM), np.float16)
specter_done = open_npy('specter_text_done.npy', (N_ROWS,), np.uint8)
pending = sorted(row for row in target_rows if not specter_done[row])
print('SPECTER pending:', len(pending), 'dimension:', SPECTER_DIM)
for start in range(0, len(pending), TEXT_BATCH):
    ids = pending[start:start + TEXT_BATCH]
    texts = [str(record_by_row[row].get('caption') or '') for row in ids]
    try:
        batch = specter_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.inference_mode():
            vectors = normalized(specter(**batch).last_hidden_state[:, 0]).cpu().numpy().astype(np.float16)
        specter_text[ids] = vectors
        specter_done[ids] = 1
    except Exception as exc:
        log_error('specter_text', ids, exc)
        raise
    if start % (TEXT_BATCH * 50) == 0:
        specter_text.flush(); specter_done.flush()
        print(f'SPECTER {start + len(ids):,}/{len(pending):,}')
specter_text.flush(); specter_done.flush()
del specter
gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter2_base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SPECTER pending: 90450 dimension: 768


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

SPECTER 64/90,450
SPECTER 3,264/90,450
SPECTER 6,464/90,450
SPECTER 9,664/90,450
SPECTER 12,864/90,450
SPECTER 16,064/90,450
SPECTER 19,264/90,450
SPECTER 22,464/90,450
SPECTER 25,664/90,450
SPECTER 28,864/90,450
SPECTER 32,064/90,450
SPECTER 35,264/90,450
SPECTER 38,464/90,450
SPECTER 41,664/90,450
SPECTER 44,864/90,450
SPECTER 48,064/90,450
SPECTER 51,264/90,450
SPECTER 54,464/90,450
SPECTER 57,664/90,450
SPECTER 60,864/90,450
SPECTER 64,064/90,450
SPECTER 67,264/90,450
SPECTER 70,464/90,450
SPECTER 73,664/90,450
SPECTER 76,864/90,450
SPECTER 80,064/90,450
SPECTER 83,264/90,450
SPECTER 86,464/90,450
SPECTER 89,664/90,450


## SigLIP2 caption embeddings

SigLIP2 supplies the only directly aligned representation across caption and image modalities. The same model remains loaded for the subsequent image pass.

In [6]:
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL, token=HF_TOKEN)
siglip = AutoModel.from_pretrained(SIGLIP_MODEL, token=HF_TOKEN, dtype=DTYPE).eval().to(DEVICE)
with torch.inference_mode():
    probe = siglip_processor(text=['astronomy figure'], padding='max_length', truncation=True, return_tensors='pt')
    probe = {k: v.to(DEVICE) for k, v in probe.items()}
    SIGLIP_DIM = int(normalized(siglip.get_text_features(**probe)).shape[-1])
siglip_text = open_npy('siglip_text_f16.npy', (N_ROWS, SIGLIP_DIM), np.float16)
siglip_text_done = open_npy('siglip_text_done.npy', (N_ROWS,), np.uint8)
pending = sorted(row for row in target_rows if not siglip_text_done[row])
print('SigLIP text pending:', len(pending), 'dimension:', SIGLIP_DIM)
for start in range(0, len(pending), TEXT_BATCH):
    ids = pending[start:start + TEXT_BATCH]
    texts = [str(record_by_row[row].get('caption') or '') for row in ids]
    try:
        batch = siglip_processor(text=texts, padding='max_length', truncation=True, return_tensors='pt')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.inference_mode():
            vectors = normalized(siglip.get_text_features(**batch)).cpu().numpy().astype(np.float16)
        siglip_text[ids] = vectors
        siglip_text_done[ids] = 1
    except Exception as exc:
        log_error('siglip_text', ids, exc)
        raise
    if start % (TEXT_BATCH * 50) == 0:
        siglip_text.flush(); siglip_text_done.flush()
        print(f'SigLIP text {start + len(ids):,}/{len(pending):,}')
siglip_text.flush(); siglip_text_done.flush()

preprocessor_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/329 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP text pending: 90450 dimension: 768
SigLIP text 64/90,450
SigLIP text 3,264/90,450
SigLIP text 6,464/90,450
SigLIP text 9,664/90,450
SigLIP text 12,864/90,450
SigLIP text 16,064/90,450
SigLIP text 19,264/90,450
SigLIP text 22,464/90,450
SigLIP text 25,664/90,450
SigLIP text 28,864/90,450
SigLIP text 32,064/90,450
SigLIP text 35,264/90,450
SigLIP text 38,464/90,450
SigLIP text 41,664/90,450
SigLIP text 44,864/90,450
SigLIP text 48,064/90,450
SigLIP text 51,264/90,450
SigLIP text 54,464/90,450
SigLIP text 57,664/90,450
SigLIP text 60,864/90,450
SigLIP text 64,064/90,450
SigLIP text 67,264/90,450
SigLIP text 70,464/90,450
SigLIP text 73,664/90,450
SigLIP text 76,864/90,450
SigLIP text 80,064/90,450
SigLIP text 83,264/90,450
SigLIP text 86,464/90,450
SigLIP text 89,664/90,450


## One-pass SigLIP2, DINOv2, and pHash image extraction

The Hugging Face split is streamed exactly once. UUID matching protects against row-order changes. Completed rows are skipped, so restored partial caches can resume, although the stream still has to advance past earlier records.

In [7]:
dino_processor = AutoImageProcessor.from_pretrained(DINO_MODEL, token=HF_TOKEN, use_fast=False)
dino = AutoModel.from_pretrained(DINO_MODEL, token=HF_TOKEN, dtype=DTYPE).eval().to(DEVICE)
DINO_DIM = int(dino.config.hidden_size)
siglip_image = open_npy('siglip_image_f16.npy', (N_ROWS, SIGLIP_DIM), np.float16)
dino_image = open_npy('dino_image_f16.npy', (N_ROWS, DINO_DIM), np.float16)
image_done = open_npy('image_done.npy', (N_ROWS,), np.uint8)
phash = open_npy('phash_u64.npy', (N_ROWS,), np.uint64)

def image_phash(image):
    gray = ImageOps.exif_transpose(image).convert('L').resize((32, 32), Image.Resampling.LANCZOS)
    coeff = dctn(np.asarray(gray, dtype=np.float32), type=2, norm='ortho')[:8, :8]
    flat = coeff.ravel()
    bits = flat > np.median(flat[1:])
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return np.uint64(value)

def strip_png_metadata(raw):
    # Scientific pixels do not depend on ICC/text chunks. Removing these also avoids
    # Pillow's decompressed-metadata safety limit on a few otherwise valid PNGs.
    signature = b'\x89PNG\r\n\x1a\n'
    if not raw.startswith(signature):
        return raw
    blocked = {b'iCCP', b'tEXt', b'zTXt', b'iTXt'}
    output = bytearray(signature)
    position = len(signature)
    while position + 12 <= len(raw):
        length = int.from_bytes(raw[position:position + 4], 'big')
        end = position + 12 + length
        if end > len(raw):
            return raw
        chunk_type = raw[position + 4:position + 8]
        if chunk_type not in blocked:
            output.extend(raw[position:end])
        position = end
        if chunk_type == b'IEND':
            break
    return bytes(output)

def decode_stream_image(value):
    if isinstance(value, Image.Image):
        return ImageOps.exif_transpose(value).convert('RGB')
    if isinstance(value, dict):
        raw = value.get('bytes')
        if raw is None and value.get('path'):
            raw = Path(value['path']).read_bytes()
    elif isinstance(value, (bytes, bytearray, memoryview)):
        raw = bytes(value)
    else:
        raise TypeError(f'Unsupported streamed image value: {type(value)}')
    if raw is None:
        raise ValueError('Streamed image has neither bytes nor a readable path')
    try:
        with Image.open(io.BytesIO(raw)) as image:
            image.load()
            return ImageOps.exif_transpose(image).convert('RGB')
    except (ValueError, SyntaxError, OSError):
        cleaned = strip_png_metadata(raw)
        if cleaned == raw:
            raise
        with Image.open(io.BytesIO(cleaned)) as image:
            image.load()
            return ImageOps.exif_transpose(image).convert('RGB')

remaining = {row for row in target_rows if not image_done[row]}
print('Image rows pending:', len(remaining), 'SigLIP dim:', SIGLIP_DIM, 'DINO dim:', DINO_DIM)
stream = load_dataset(HF_DATASET, split=HF_SPLIT, streaming=True, token=HF_TOKEN)
# Disable datasets' eager Pillow decoding so malformed/metadata-heavy files can be
# handled per row instead of terminating the iterator before our try/except block.
stream = stream.cast_column('image', HFImage(decode=False))
available_columns = set(getattr(stream, 'column_names', None) or stream.features.keys())
columns = ['image'] + (['UUID'] if 'UUID' in available_columns else [])
stream = stream.select_columns(columns)
processed = 0
seen_stream_rows = 0
started = time.time()
for batch in stream.iter(batch_size=IMAGE_BATCH):
    images, ids = [], []
    batch_images = batch['image']
    batch_uuids = batch.get('UUID', [''] * len(batch_images))
    for offset, (image, uuid) in enumerate(zip(batch_images, batch_uuids)):
        fallback_row = seen_stream_rows + offset
        row = uuid_to_row.get(str(uuid or ''), fallback_row)
        if row not in remaining:
            continue
        try:
            images.append(decode_stream_image(image))
            ids.append(row)
        except Exception as exc:
            log_error('image_decode', row, exc)
    seen_stream_rows += len(batch_images)
    if not ids:
        continue
    try:
        sig_batch = siglip_processor(images=images, padding='max_length', max_num_patches=256, return_tensors='pt')
        sig_batch = {k: v.to(DEVICE) for k, v in sig_batch.items()}
        dino_batch = dino_processor(images=images, return_tensors='pt')
        dino_batch = {k: v.to(DEVICE) for k, v in dino_batch.items()}
        with torch.inference_mode():
            sig_vec = normalized(siglip.get_image_features(**sig_batch)).cpu().numpy().astype(np.float16)
            dino_out = dino(**dino_batch)
            dino_vec = normalized(dino_out.last_hidden_state[:, 0]).cpu().numpy().astype(np.float16)
        siglip_image[ids] = sig_vec
        dino_image[ids] = dino_vec
        phash[ids] = np.asarray([image_phash(image) for image in images], dtype=np.uint64)
        image_done[ids] = 1
        remaining.difference_update(ids)
        processed += len(ids)
    except Exception as exc:
        log_error('image_embedding', ids, exc)
        raise
    if processed and processed % CHECKPOINT_EVERY < len(ids):
        siglip_image.flush(); dino_image.flush(); phash.flush(); image_done.flush()
        elapsed = max(time.time() - started, 1)
        print(f'Images {processed:,} this run; {len(remaining):,} remaining; {processed/elapsed:.2f}/s')
    if not remaining:
        break
siglip_image.flush(); dino_image.flush(); phash.flush(); image_done.flush()
del siglip, dino
gc.collect(); torch.cuda.empty_cache()
print('Stream rows visited:', seen_stream_rows, 'remaining targets:', len(remaining))

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Image rows pending: 90450 SigLIP dim: 768 DINO dim: 384


...

Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Images 518 this run; 89,932 remaining; 10.22/s
Images 1,026 this run; 89,424 remaining; 10.55/s
Images 1,539 this run; 88,911 remaining; 10.62/s
Images 2,050 this run; 88,400 remaining; 10.55/s
Images 2,567 this run; 87,883 remaining; 10.48/s
Images 3,076 this run; 87,374 remaining; 10.60/s
Images 3,585 this run; 86,865 remaining; 10.48/s
Images 4,098 this run; 86,352 remaining; 10.35/s
Images 4,615 this run; 85,835 remaining; 10.50/s
Images 5,125 this run; 85,325 remaining; 10.44/s
Images 5,635 this run; 84,815 remaining; 10.35/s
Images 6,144 this run; 84,306 remaining; 10.43/s
Images 6,662 this run; 83,788 remaining; 10.34/s
Images 7,170 this run; 83,280 remaining; 10.31/s
Images 7,686 this run; 82,764 remaining; 10.35/s
Images 8,196 this run; 82,254 remaining; 10.30/s
Images 8,708 this run; 81,742 remaining; 10.27/s
Images 9,218 this run; 81,232 remaining; 10.31/s
Images 9,734 this run; 80,716 remaining; 10.35/s
Images 10,241 this run; 80,209 remaining; 10.43/s
Images 10,758 this ru

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/adsabs/AstroCLIMB/resolve/68e8d0ccb44be6bb86a77ccb90fdc8ba735dcdcb/data/train-00048-of-00114.parquet
Retrying in 1s [Retry 1/5].


Images 38,402 this run; 52,048 remaining; 9.98/s
Images 38,917 this run; 51,533 remaining; 9.98/s
Images 39,425 this run; 51,025 remaining; 9.97/s
Images 39,936 this run; 50,514 remaining; 9.96/s
Images 40,453 this run; 49,997 remaining; 9.96/s
Images 40,967 this run; 49,483 remaining; 9.95/s
Images 41,478 this run; 48,972 remaining; 9.93/s
Images 41,987 this run; 48,463 remaining; 9.93/s
Images 42,502 this run; 47,948 remaining; 9.93/s
Images 43,012 this run; 47,438 remaining; 9.92/s
Images 43,526 this run; 46,924 remaining; 9.94/s
Images 44,038 this run; 46,412 remaining; 9.94/s
Images 44,546 this run; 45,904 remaining; 9.93/s
Images 45,063 this run; 45,387 remaining; 9.94/s
Images 45,572 this run; 44,878 remaining; 9.93/s
Images 46,085 this run; 44,365 remaining; 9.93/s
Images 46,599 this run; 43,851 remaining; 9.93/s
Images 47,105 this run; 43,345 remaining; 9.93/s
Images 47,618 this run; 42,832 remaining; 9.93/s
Images 48,131 this run; 42,319 remaining; 9.92/s
Images 48,646 this r

## Coverage and integrity gate

Every target needs all four applicable cached representations. This cell fails if extraction silently missed a row or produced a non-finite vector.

In [8]:
target_ids = np.asarray(sorted(target_rows), dtype=np.int64)
coverage = {
    'specter_text': int(specter_done[target_ids].sum()),
    'siglip_text': int(siglip_text_done[target_ids].sum()),
    'image': int(image_done[target_ids].sum()),
}
print('Coverage:', coverage, 'expected each:', len(target_ids))
assert coverage['specter_text'] == len(target_ids)
assert coverage['siglip_text'] == len(target_ids)
assert coverage['image'] == len(target_ids)
for name, array in (
    ('specter_text', specter_text), ('siglip_text', siglip_text),
    ('siglip_image', siglip_image), ('dino_image', dino_image),
):
    sample_ids = target_ids[::max(1, len(target_ids)//1000)]
    assert np.isfinite(np.asarray(array[sample_ids], dtype=np.float32)).all(), f'Non-finite {name}'
    norms = np.linalg.norm(np.asarray(array[sample_ids], dtype=np.float32), axis=1)
    assert np.all((norms > 0.90) & (norms < 1.10)), f'Bad normalized vectors in {name}: {norms.min()}..{norms.max()}'
print('Integrity checks passed.')

Coverage: {'specter_text': 90450, 'siglip_text': 90450, 'image': 90450} expected each: 90450
Integrity checks passed.


## Save cache manifest

Save the complete `astroclimb_03` directory as a private Kaggle Dataset. Large array checksums are intentionally omitted to avoid another full read; shapes, dtypes, coverage, model IDs, source manifest checksums, and a target-row checksum are recorded.

In [9]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

target_digest = hashlib.sha256(target_ids.tobytes()).hexdigest()
summary = {
    'source': {
        'metadata_index': str(INDEX_PATH), 'hf_dataset': HF_DATASET, 'hf_split': HF_SPLIT,
        'train_manifest': str(TRAIN_MANIFEST), 'validation_manifest': str(VALIDATION_MANIFEST),
        'train_manifest_sha256': sha256_file(TRAIN_MANIFEST),
        'validation_manifest_sha256': sha256_file(VALIDATION_MANIFEST),
    },
    'models': {'specter': SPECTER_MODEL, 'siglip': SIGLIP_MODEL, 'dino': DINO_MODEL},
    'dimensions': {'specter': SPECTER_DIM, 'siglip': SIGLIP_DIM, 'dino': DINO_DIM},
    'storage_dtype': 'float16', 'metadata_rows': N_ROWS,
    'target_rows': len(target_rows), 'train_target_rows': len(train_rows),
    'validation_target_rows': len(validation_rows), 'target_rows_sha256': target_digest,
    'coverage': coverage,
    'files': {},
}
for path in sorted(OUTPUT_DIR.glob('*.npy')):
    array = np.load(path, mmap_mode='r')
    summary['files'][path.name] = {'shape': list(array.shape), 'dtype': str(array.dtype), 'bytes': path.stat().st_size}
with (OUTPUT_DIR / 'representation_cache_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2, sort_keys=True)
print(json.dumps(summary, indent=2, sort_keys=True))
print('Total cache size GiB:', sum(p.stat().st_size for p in OUTPUT_DIR.iterdir()) / 1024**3)

{
  "coverage": {
    "image": 90450,
    "siglip_text": 90450,
    "specter_text": 90450
  },
  "dimensions": {
    "dino": 384,
    "siglip": 768,
    "specter": 768
  },
  "files": {
    "dino_image_f16.npy": {
      "bytes": 72371072,
      "dtype": "float16",
      "shape": [
        94233,
        384
      ]
    },
    "image_done.npy": {
      "bytes": 94361,
      "dtype": "uint8",
      "shape": [
        94233
      ]
    },
    "phash_u64.npy": {
      "bytes": 753992,
      "dtype": "uint64",
      "shape": [
        94233
      ]
    },
    "siglip_image_f16.npy": {
      "bytes": 144742016,
      "dtype": "float16",
      "shape": [
        94233,
        768
      ]
    },
    "siglip_text_done.npy": {
      "bytes": 94361,
      "dtype": "uint8",
      "shape": [
        94233
      ]
    },
    "siglip_text_f16.npy": {
      "bytes": 144742016,
      "dtype": "float16",
      "shape": [
        94233,
        768
      ]
    },
    "specter_text_done.npy": {
      "by

## Handoff

Download `representation_cache_summary.json`, preserve the executed notebook, and save `/kaggle/working/astroclimb_03` as a private Kaggle Dataset. The next notebook will create symmetric pair features, encode the Kaggle train/test objects, and evaluate E03–E06 without repeating this Hugging Face image pass.